In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

plt.rcParams["axes.grid"] = False

#sns.set_theme(style="white") 
plt.rcdefaults()

In [3]:
## df = pd.read_csv("db_bereinigt.csv")  # alte csv

#df = pd.read_csv("db_bereinigt_fertig.csv") # aktuelle Datenbank Stand 06.07.2026

df = pd.read_csv("../Daten/db_bereinigt_final.csv") # aktuelle Datenbank Stand 13.07.2026

In [ ]:
'''
# Daten aus Datenbank einlesen

from sqlalchemy import create_engine
import os
from dotenv import load_dotenv

load_dotenv()
db_password = os.getenv("DB_PASSWORD")

# Verbindungsdaten anpassen
db_user = 'neondb_owner'
db_host = 'ep-summer-band-aselkm9n-pooler.c-4.eu-central-1.aws.neon.tech'
db_port = '5432'
db_name = 'neondb'

# Engine erstellen
engine = create_engine(f'postgresql+psycopg2://{db_user}:{db_password}@{db_host}:{db_port}/{db_name}')

# SQL-Query definieren und Daten in den DataFrame laden
query = "SELECT * FROM ashrae_consolidado"
df = pd.read_sql(query, con=engine)

# Erste Zeilen zur Kontrolle anzeigen
display(df.head())
'''

In [ ]:
print(type(df))

In [ ]:
df["region"].value_counts()

In [ ]:
df

# Target Clo

In [ ]:
# Target Kleidungsisolationswert
#df_ml_clo = df[['outdoor_air_temperature', 'air_temperature', 'relative_humidity', 'season', 'country', 'building_type', 'air_speed', 'clothing_ensemble_insulation']] #   Basis 0,13 MAE
#df_ml_clo = df[['outdoor_air_temperature', 'air_temperature', 'relative_humidity', 'season', 'country', 'building_type', 'air_speed', 'metabolic_rate', 'clothing_ensemble_insulation']]   # MAE 0,12
#df_ml_clo = df[['outdoor_air_temperature', 'air_temperature', 'relative_humidity', 'season', 'country', 'building_type', 'air_speed', 'metabolic_rate', 'climate', 'clothing_ensemble_insulation']]

# ggf. Modell für Dashboard
#df_ml_clo = df[['outdoor_air_temperature', 'air_temperature', 'relative_humidity', 'season', 'country', 'building_type', 'air_speed', 'metabolic_rate', 'climate_zone', 'cooling_type', 'clothing_ensemble_insulation']]
df_ml_clo = df[['outdoor_air_temperature', 'air_temperature', 'relative_humidity', 'season', 'building_type', 'air_speed', 'metabolic_rate', 'climate_zone', 'cooling_type', 'clothing_ensemble_insulation']]

In [ ]:
df_ml_clo

In [ ]:
df_ml_clo['cooling_type'] = df_ml_clo['cooling_type'].replace(['unknown', 'Unknown', 'UNKNOWN', 'nan', 'None'], np.nan)

In [ ]:
df_ml_clo_clean = df_ml_clo.dropna()


In [ ]:
df_ml_clo_clean.info()

In [ ]:
# Der clo-Wert ist ein kontinuierlicher Wert!!!!!

In [ ]:
# Korrelationsmatrix

#corr_matrix = df_ml_clo_clean.drop(columns=['cooling_type', 'climate_zone', 'building_type', 'country', 'season', 'clothing_ensemble_insulation']).corr()
corr_matrix = df_ml_clo_clean.drop(columns=['cooling_type', 'climate_zone', 'building_type', 'season', 'clothing_ensemble_insulation']).corr()

# 2. Grafik (Heatmap) erstellen
plt.figure(figsize=(8, 6))

# sns.heatmap zeichnet das Diagramm
# annot=True schreibt die exakten Zahlenwerte in die Kästchen
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm',  # Blau = negative Korrelation, Rot = positive Korrelation
    fmt=".2f",        # Auf 2 Nachkommastellen runden
    linewidths=0.5,   # Dünne Trennlinien zwischen den Kästchen
    vmin=-1, vmax=1   # Die Farbskala exakt von -1 bis +1 begrenzen
)

plt.title('Korrelationsmatrix der Features', fontsize=14, pad=15)
plt.tight_layout()
plt.show()


In [ ]:
#sns.pairplot(data=df_ml_clean, hue='clothing_ensemble_insulation', palette="Set2")

In [ ]:
# airspeed sollte skaliert werden
# ggf. weitere Spalten bzgl. der Werte, weniger für die Verteilung



In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, PowerTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error

from sklearn.preprocessing import OneHotEncoder, TargetEncoder

X = df_ml_clo_clean.drop(columns=['clothing_ensemble_insulation'])
y = df_ml_clo_clean['clothing_ensemble_insulation']

categorial_features = [
    "season",
#    "country",
    "building_type",
    "cooling_type",
    "climate_zone"
]

#climate_features = ["climate"]    # vorher wegen zu vieler Einträge
#climate_features = ["climate_zone"]    # vorher wegen zu vieler Einträge

feature_names = df_ml_clo_clean.drop(columns=['clothing_ensemble_insulation']).columns.tolist()

# Spalten aufteilen: Nur 'feature_0' und 'feature_1' sollen skaliert werden
#columns_to_scale = ['air_speed']
columns_to_scale = ['air_speed', 'metabolic_rate']
standard_columns = ['outdoor_air_temperature', 'air_temperature', 'relative_humidity']
other_columns = [col for col in feature_names if col not in columns_to_scale]

# 2. Train-Test-Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)


# 3. ColumnTransformer für gezielte Skalierung
preprocessor = ColumnTransformer(
    transformers=[
        ('power_transform', PowerTransformer(method='yeo-johnson'), columns_to_scale),
        ('standard_scale', StandardScaler(), standard_columns),
        ('cat_encoder', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), categorial_features)#,   # sparse_output für GradientBoostingRegrressor
#        ('climate_encoder', TargetEncoder(smooth="auto", cv=5, random_state=42), climate_features)
    ],
    remainder='passthrough' # Verbleibende Spalten bleiben unverändert
)

# manuelles Logarithmieren des targets
#y_train_log = np.log1p(y_train)
#y_test_log = np.log1p(y_test)

In [ ]:
# Rausschreiben für Isolation Forest zur Anomalieerkennung
#X_train.to_csv('X_train.csv', index=False, sep=';')
#y_train.to_csv('y_train.csv', index=False, sep=';')

#X_test.to_csv('X_test.csv', index=False, sep=';')
#y_test.to_csv('y_test.csv', index=False, sep=';')

In [ ]:
# Einlesen der Train und Testsets aus der Anomalieerkennung
# R^2 Werte geringfügig schlechter, aber keien anormalen Werte mehr im Trainset
#X_train = pd.read_csv('X_train_cleaned.csv', sep=';')
#y_train = pd.read_csv('y_train_cleaned.csv', sep=';')

# Trainset sollte nur nzgl. Punkten angefasst werden die physikalisch unlogisch sind!!!
# Somit sollte jeder Punkt der als anormal deklariert wurde kontrolliert werden!!!

In [ ]:
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.preprocessing import PolynomialFeatures
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor  
from sklearn.svm import SVR

# Definition der Algorithmen
models = {
    "Linear Regression": LinearRegression(),    #   quasi keine Hyperparameter nur über Regularisierungen oder durch Feature Engineering zu verbessern
    "Linear Regression (mit Polynomen - 2ten Grades)": Pipeline([('poly', PolynomialFeatures(degree=2, interaction_only=True)), ('lr', LinearRegression())]),

     # --- HIER SIND DIE NEUEN LASSO MODELLE ---
##    "Lasso Regression": Lasso(alpha=0.1, max_iter=10000, random_state=42),
##    "Lasso Regression (mit Polynomen)": Pipeline([
##        ('poly', PolynomialFeatures(degree=2, interaction_only=True)), 
##        ('lr', Lasso(alpha=0.1, max_iter=10000, random_state=42))
##    ]),

    "Ridge Regression": Ridge(alpha=0.001),
    
##        "Ridge Regression (mit Polynomen, alpha=10.0)": Pipeline([
##        ('poly', PolynomialFeatures(degree=2, interaction_only=True)), 
##        ('lr', Ridge(alpha=0.01))
##    ]),

    "Support Vector Regression": SVR(kernel='rbf'), # nicht effektiv bei vielen Datensätzen!!!

    # SVR mit höherer Toleranz für komplexe Muster (höheres C)
#    "SVR (C=10, epsilon=0.1)": SVR(kernel='rbf', C=10.0, epsilon=0.1), # keine Verbesserung
    
    # SVR mit sehr feiner, lokaler Anpassung (höheres Gamma)
##    "SVR (C=100, gamma=0.1)": SVR(kernel='rbf', C=100.0, gamma=0.1),  # extrem lange Simulationszeit!!! abgebrochen
    
    # Alternativer Kernel für rein lineare/polynomiale Trends (falls RBF nicht performt)
##    "SVR (Linear-Kernel)": SVR(kernel='linear', C=1.0),  # extrem lange Simulationszeit!!! abgebrochen

    #"Random Forest": RandomForestRegressor(n_estimators=100, random_state=42, max_depth=3, min_samples_leaf=5),

#    "Random Forest (Flach)": RandomForestRegressor(n_estimators=100, random_state=42, max_depth=3, min_samples_leaf=5),
    
    # Variante 1: Tieferer Wald für komplexere Muster
#    "Random Forest (Medium)": RandomForestRegressor(n_estimators=150, random_state=42, max_depth=8, min_samples_leaf=4),
    
    # Variante 2: Sehr flexibler Wald, bei dem max_features die Overfitting-Bremse zieht
    "Random Forest (Deep & Random)": RandomForestRegressor(n_estimators=200, random_state=42, max_depth=15, min_samples_leaf=2, max_features='sqrt'),
#    "Random Forest (Deeper & Random)": RandomForestRegressor(n_estimators=200, random_state=42, max_depth=20, min_samples_leaf=2, max_features='sqrt'),
#    "Random Forest (Deeper & Random)": RandomForestRegressor(n_estimators=1000, random_state=42, max_depth=15, min_samples_leaf=2, max_features='sqrt'),

    "HistGradientBoosting": HistGradientBoostingRegressor(
        max_iter=200,          # Entspricht n_estimators bei XGBoost (Anzahl Bäume) 150, 200
        max_depth=10,           # Maximale Tiefe der Bäume   5, 10
        learning_rate=0.15,    # Lernrate / Schrittweite (Overfitting-Bremse)   0.05, 0.15
        random_state=42
    )
}

# Liste, um die Ergebnisse für das Plotten zu speichern
plot_data = []

# Schleife über alle Modelle
for name, model in models.items():
    # Pipeline aus Vorverarbeitung und aktuellem Modell erstellen
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('regressor', model)
    ])
    
    # Modell trainieren
 #   pipeline.fit(X_train, y_train_log)
    pipeline.fit(X_train, y_train)
    
    # Vorhersagen für Training und Test generieren
#    y_train_pred_log = pipeline.predict(X_train)
#    y_test_pred_log = pipeline.predict(X_test)
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
#    y_train_pred = np.expm1(y_train_pred_log)
#    y_test_pred = np.expm1(y_test_pred_log)

    # Metriken berechnen
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)
    
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    
    # Overfitting-Check: Differenz der R²-Werte
    r2_diff = r2_train - r2_test
    
    print(f"=== {name} ===")
    #print(f"  Train: R² = {r2_train:.4f} | RMSE = {rmse_train:.2f}")
    #print(f"  Test:  R² = {r2_test:.4f}  | RMSE = {rmse_test:.2f}")
    print(f"  Train: R² = {r2_train:.4f} | RMSE = {rmse_train:.2f} | MAE = {mae_train:.2f}")
    print(f"  Test:  R² = {r2_test:.4f} | RMSE = {rmse_test:.2f} | MAE = {mae_test:.2f}")

    print(f"Delta R1: {r2_diff}")

    plot_data.append({"Modell": name, "Datensatz": "Training", "R²-Wert": r2_train})
    plot_data.append({"Modell": name, "Datensatz": "Test", "R²-Wert": r2_test})

    # Warnung ausgeben, wenn das Modell auf den Trainingsdaten deutlich besser ist
    if r2_diff > 0.10:
        print(f"  ⚠️ Warnung: Mögliches Overfitting erkannt! (R²-Differenz: {r2_diff:.4f})")
    elif r2_diff < -0.05:
        print(f"  ℹ️ Modell generalisiert ungewöhnlich gut oder Testset ist zu klein.")
    else:
        print(f"  ✅ Modell ist stabil (Gute Balance zwischen Train und Test).")
    print("-" * 40)


# Es konnte kein Unterschid zwischen logartimierten und nicht logarithmierten Targetwerten festgestellt

\(R^{2}\) < 0,30 (Schwach): Das Modell hat kaum Vorhersagekraft. Es übersieht wichtige Muster oder die Daten sind zu verrauscht.<br>
\(R^{2}\) von 0,30 bis 0,50 (Moderat/Akzeptabel): Typisch für komplexe, menschliche oder biologische Systeme. Das Modell hat reale Muster erkannt, liegt aber noch oft daneben.<br>
\(R^{2}\) von 0,50 bis 0,70 (Gut): Ein starkes Ergebnis in der Praxis mit realen Felddaten.<br>
\(R^{2}\) > 0,70 (Sehr gut / Exakt): Wird meist nur in der Physik, bei technischen Anlagen oder im Labor erreicht, wo Prozesse exakt berechenbaren Regeln folgen.<br>

0.52 heißt, dass das Modell zu 52% aus den Features zu erklären ist und zu 48% durch nicht beobachtete Features oder dem Zufdall!

In [ ]:
# Ergebnisse in einen DataFrame umwandeln
df_results = pd.DataFrame(plot_data)

# Grafik-Größe festlegen
#plt.figure(figsize=(10, 6))

plt.figure(figsize=(11, 7))
sns.set_theme(style="whitegrid") # Sorgt für ein sauberes, modernes Design

# Balkendiagramm erstellen (trennt automatisch nach Training und Test)
sns.barplot(
    data=df_results, 
    y="Modell", 
    x="R²-Wert", 
    hue="Datensatz", 
    palette="muted"
)

# Optische Verschönerungen
plt.title("Vergleich der R²-Werte der Regressionsmodelle - Target clo", fontsize=14, fontweight='bold', pad=15)
plt.xlabel("R²-Wert (erklärbare Varianz)", fontsize=12, labelpad=10)
plt.ylabel("Algorithmus", fontsize=12, labelpad=10)
plt.xlim(0, 1.0) # Setzt die Achse von 0 bis 1 (perfekt)
plt.grid(axis='x', linestyle='--', alpha=0.7) # Hintergrundraster

# Werte direkt an den Balken anzeigen (optional, ab matplotlib 3.4+)
for container in plt.gca().containers:
    plt.gca().bar_label(container, fmt='%.2f', padding=5)

plt.legend(title="Datensatz", loc="lower right", frameon=True)

plt.tight_layout()
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import r2_score  # <--- NEU: R2-Score importieren
from sklearn.pipeline import Pipeline

# Dictionary, um die Ergebnisse für die Grafik zu speichern
r2_results = {}  # <--- Umbenannt für R2

# Die Schleife trainiert die Modelle und speichert den R2-Wert
for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model),  # <--- Von 'classifier' zu 'regressor' geändert
        ]
    )

    # Trainieren & Vorhersagen
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # R2-Score berechnen und speichern
    score = r2_score(y_test, y_pred)  # <--- NEU: R2 statt F1
    r2_results[name] = score

# --- Grafik erstellen ---
plt.figure(figsize=(10, 6))

# Farben für die einzelnen Balken definieren
farben = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

# Balkendiagramm zeichnen
bars = plt.bar(
    r2_results.keys(),
    r2_results.values(),
    color=farben,
    edgecolor="black",
    width=0.6,
)

# Werte oben auf die Balken schreiben
for bar in bars:
    height = bar.get_height()
    # Bei negativen R2-Werten wird das Label unter den Balken gesetzt
    va_dir = "bottom" if height >= 0 else "top"
    offset = 0.01 if height >= 0 else -0.01

    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + offset,
        f"{height:.2f}",
        ha="center",
        va=va_dir,
        fontsize=11,
        fontweight="bold",
    )

# Diagramm-Details anpassen
plt.title("Modellvergleich: R²-Wert (Bestimmtheitsmaß)", fontsize=14, pad=15)
plt.ylabel("R²-Score", fontsize=12)

# Dynamische Y-Achse, falls ein Modell sehr schlechte (negative) R2-Werte liefert
min_y = min(0.0, min(r2_results.values()) - 0.1)
plt.ylim(min_y, 1.1)  # Platz für die Textlabels oben lassen

plt.grid(axis="y", linestyle="--", alpha=0.7)

# Grafik anzeigen
plt.tight_layout()
plt.xticks(rotation=90)
plt.show()


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import mean_absolute_error  # <--- NEU: MAE importieren
from sklearn.pipeline import Pipeline

# Dictionary, um die Ergebnisse für die Grafik zu speichern
mae_results = {}

# Die Schleife trainiert die Modelle und speichert den MAE-Wert
for name, model in models.items():
    pipeline = Pipeline(
        steps=[
            ("preprocessor", preprocessor),
            ("regressor", model),
        ]
    )

    # Trainieren & Vorhersagen
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)

    # MAE berechnen und speichern
    score = mean_absolute_error(y_test, y_pred)  # <--- NEU: MAE statt R2
    mae_results[name] = score

# --- Grafik erstellen ---
plt.figure(figsize=(10, 6))

# Farben für die einzelnen Balken definieren
farben = ["#4C72B0", "#55A868", "#C44E52", "#8172B2"]

# Balkendiagramm zeichnen
bars = plt.bar(
    mae_results.keys(),
    mae_results.values(),
    color=farben,
    edgecolor="black",
    width=0.6,
)

# Höchsten MAE-Wert ermitteln, um dynamisch Platz für die Labels zu schaffen
max_mae = max(mae_results.values())
text_offset = max_mae * 0.02  # 2% der Höhe als Abstand über dem Balken

# Werte oben auf die Balken schreiben
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width() / 2.0,
        height + text_offset,
        f"{height:.3f}",  # <--- .3f für präzisere Fehlerwerte (3 Nachkommastellen)
        ha="center",
        va="bottom",
        fontsize=11,
        fontweight="bold",
    )

# Diagramm-Details anpassen
plt.title(
    "Modellvergleich: Mean Absolute Error (MAE) - Weniger ist besser",
    fontsize=14,
    pad=15,
)
plt.ylabel("Mittlerer absoluter Fehler (MAE)", fontsize=12)

# Y-Achse startet starr bei 0 und lässt oben 15% Puffer für die Text-Beschriftungen
plt.ylim(0, max_mae * 1.15)

plt.grid(axis="y", linestyle="--", alpha=0.7)

# Grafik anzeigen
plt.tight_layout()
plt.xticks(rotation=90)
plt.show()


### Gridsearch - Anomaliefilter - log. Skalierung clo

Logarithmieren des Targets oben ausprobiert => keine Änderung der Genauigkeit!

In [ ]:
'''
from sklearn.model_selection import GridSearchCV

base_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Nutzt deinen bestehenden preprocessor
    ('regressor', HistGradientBoostingRegressor(loss='absolute_error', random_state=42))
])

# Hyperparameter-Gitter (Param_Grid) definieren

param_grid = {
    'regressor__learning_rate': [0.01, 0.05, 0.1],     # Lernrate des Modells
    'regressor__max_iter': [50, 100, 150, 250],              # Anzahl der Boosting-Schritte (Bäume)
    'regressor__max_leaf_nodes': [15, 31, 63],          # Maximale Anzahl an Blättern pro Baum
    'regressor__min_samples_leaf': [10, 20, 30],        # Mindestanzahl an Datenpunkten pro Blatt
    'regressor__l2_regularization': [0.0, 0.1, 1.0],     # L2-Regularisierung zur Vermeidung von Overfitting
    'regressor__max_depth': [5, 12 ,20 ,50],
    'regressor__early_stopping': [True, None]
}

# 4. GridSearchCV konfigurieren
# cv=5 bedeutet 5-fache Kreuzvalidierung
# n_jobs=-1 nutzt alle verfügbaren CPU-Kerne für maximale Geschwindigkeit
grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring='neg_mean_absolute_error',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)   # kein log verwendet

print("\n--- Grid Search abgeschlossen ---")
print(f"Beste Hyperparameter: {grid_search.best_params_}")
print(f"Bester CV-Score (MAE auf Log-Ebene): {-grid_search.best_score_:.4f}\n")


# --- 3. DIE BESTE PIPELINE IN DEINER EVALUATIONS-SCHLEIFE NUTZEN ---

# Wir packen das beste gefundene Modell in dein models-Dictionary
models = {
    "HistGradientBoosting (Optimized via GridSearch)": grid_search.best_estimator_
}

# Liste, um die Ergebnisse für das Plotten zu speichern
plot_data = []

# Schleife über alle Modelle (jetzt nur noch das optimierte HistGradientBoosting)
for name, pipeline in models.items():
    
    # Hinweis: .fit() wurde bereits oben in der Grid Search ausgeführt!
    # Die pipeline enthält bereits das fertig trainierte, beste Modell.
    
    # Vorhersagen für Training und Test generieren
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)
    
    # Logarithmierung rückgängig machen (wie in deiner Vorlage)
#    y_train_pred = np.expm1(y_train_pred_log)
#    y_test_pred = np.expm1(y_test_pred_log)

    # Metriken berechnen (auf den Original-Werten)
    r2_train = r2_score(y_train, y_train_pred)
    r2_test = r2_score(y_test, y_test_pred)
    
    rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
    rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

    mae_train = mean_absolute_error(y_train, y_train_pred)
    mae_test = mean_absolute_error(y_test, y_test_pred)
    
    # Overfitting-Check: Differenz der R²-Werte
    r2_diff = r2_train - r2_test
    
    print(f"=== {name} ===")
    print(f"  Train: R² = {r2_train:.4f} | RMSE = {rmse_train:.2f} | MAE = {mae_train:.2f}")
    print(f"  Test:  R² = {r2_test:.4f}  | RMSE = {rmse_test:.2f}  | MAE = {mae_test:.2f}")

    print(f"Delta R²: {r2_diff:.4f}")

    plot_data.append({"Modell": name, "Datensatz": "Training", "R²-Wert": r2_train})
    plot_data.append({"Modell": name, "Datensatz": "Test", "R²-Wert": r2_test})

    # Warnung ausgeben, wenn das Modell auf den Trainingsdaten deutlich besser ist
    if r2_diff > 0.10:
        print(f"  ⚠️ Warnung: Mögliches Overfitting erkannt! (R²-Differenz: {r2_diff:.4f})")
    elif r2_diff < -0.05:
        print(f"  ℹ️ Modell generalisiert ungewöhnlich gut oder Testset ist zu klein.")
    else:
        print(f"  ✅ Modell ist stabil (Gute Balance zwischen Train und Test).")
    print("-" * 40)
    '''

In [ ]:
'''
nach 35 min, kein besseres Ergebnis als in ursprünglicher Untersuchung!!!
--- Grid Search abgeschlossen ---
Beste Hyperparameter: {'regressor__early_stopping': True, 'regressor__l2_regularization': 0.1, 'regressor__learning_rate': 0.1, 'regressor__max_depth': 12, 'regressor__max_iter': 250, 'regressor__max_leaf_nodes': 63, 'regressor__min_samples_leaf': 30}
Bester CV-Score (MAE auf Log-Ebene): 0.1198

=== HistGradientBoosting (Optimized via GridSearch) ===
  Train: R² = 0.6526 | RMSE = 0.17 | MAE = 0.11
  Test:  R² = 0.5948  | RMSE = 0.18  | MAE = 0.12
Delta R²: 0.0578
  ✅ Modell ist stabil (Gute Balance zwischen Train und Test).
----------------------------------------
'''

### SHAP-Analyse

In [ ]:
import shap
import matplotlib.pyplot as plt

# 1. Das gewünschte Modell auswählen (hier HistGradientBoosting als Beispiel)
# Wir holen die trainierte Pipeline aus deiner Schleife oder bauen sie kurz nach:
best_model_name = "HistGradientBoosting"
#best_model_name = "Random Forest (Deep & Random)"

pipeline_to_explain = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', models[best_model_name])
])
# Erneuter Fit zur Sicherheit, falls die Schleife oben nicht die trainierte Pipeline speichert
pipeline_to_explain.fit(X_train, y_train)

# 2. Feature-Namen nach der Transformation extrahieren
# get_feature_names_out() liefert die Namen für One-Hot, TargetEncoder etc.
transformed_feature_names = pipeline_to_explain.named_steps['preprocessor'].get_feature_names_out()

# 3. Testdaten transformieren, um sie dem Explainer zu übergeben
X_test_transformed = pipeline_to_explain.named_steps['preprocessor'].transform(X_test)

# Optional: In ein DataFrame gießen für schönere SHAP-Labels
X_test_df = pd.DataFrame(X_test_transformed, columns=transformed_feature_names)

# 4. SHAP Explainer initialisieren und Werte berechnen
# TreeExplainer ist optimiert für HistGradientBoosting und RandomForest
explainer = shap.TreeExplainer(pipeline_to_explain.named_steps['regressor'])
#shap_values = explainer(X_test_df)


In [ ]:
shap_values_raw = explainer(X_test_df)

# --- NEU: Zusammenfassen der One-Hot-Spalten auf die ursprünglichen 10 Features ---
# Wir gruppieren die Spalten anhand ihres Präfixes (z.B. 'cat__season_winter' -> 'season')
original_features = ['outdoor_air_temperature', 'air_temperature', 'relative_humidity', 
#                     'season', 'country', 'building_type', 'air_speed', 
                     'season', 'building_type', 'air_speed', 
                     'metabolic_rate', 'climate_zone', 'cooling_type']

# Neue leere Arrays für die zusammengefassten Daten erstellen
collapsed_values = np.zeros((X_test_df.shape[0], len(original_features)))
collapsed_data = np.zeros((X_test_df.shape[0], len(original_features)))

for i, orig_feat in enumerate(original_features):
    # Finde alle transformierten Spaltennamen, die den Namen des Original-Features enthalten
    matching_cols = [col for col in X_test_df.columns if orig_feat in col]
    matching_indices = [X_test_df.columns.get_loc(col) for col in matching_cols]
    
    # 1. SHAP-Werte für alle zusammengehörigen Kategorien aufaddieren
    collapsed_values[:, i] = shap_values_raw.values[:, matching_indices].sum(axis=1)
    
    # 2. Den tatsächlichen Wert abbilden
    # Bei numerischen Werten nehmen wir den Wert direkt. Bei One-Hot-Kategorien setzen wir 
    # für die historische Übersicht einen Platzhalter (oder den Max-Wert) ein.
    if len(matching_indices) == 1:
        collapsed_data[:, i] = X_test_df.iloc[:, matching_indices[0]]
    else:
        # Repräsentiert die aktivierte Kategorie (wo die 1 im One-Hot-Encoding steht)
        collapsed_data[:, i] = X_test_df.iloc[:, matching_indices].max(axis=1)

# Ein neues SHAP-Explanation-Objekt mit exakt 10 Spalten bauen
from shap import Explanation
shap_values_collapsed = Explanation(
    values=collapsed_values,
    base_values=shap_values_raw.base_values,
    data=collapsed_data,
    feature_names=original_features
)

In [ ]:
print(f"\n=== Globales SHAP-Balkendiagramm für {best_model_name} ===")
plt.figure(figsize=(10, 6))

# Generiert ein sauberes, sortiertes Balkendiagramm der globalen Wichtigkeit
# max_display=15 zeigt die Top 15 wichtigsten Features an
shap.plots.bar(shap_values_collapsed, max_display=15, show=False)

plt.title(f"Globale Feature-Wichtigkeit (Absoluter SHAP-Mittelwert) - {best_model_name}", fontsize=14, pad=20)
plt.xlabel("Mittlerer absoluter SHAP-Wert (Einfluss auf die Vorhersage)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# --- GLOBALE ANALYSE (Summary Plot mit 10 komprimierten Features) ---
print(f"\n=== Globale SHAP-Analyse für {best_model_name} ===")

# 1. Eine saubere 10-Spalten-Kopie des originalen, un-transformierten X_test erstellen
X_test_summary = X_test.copy()

# 2. Textspalten in numerische Codes umwandeln, damit SHAP sie plotten kann
for col in X_test_summary.select_dtypes(include=['object', 'category']).columns:
    X_test_summary[col] = X_test_summary[col].astype('category').cat.codes

# 3. Grafik mit perfekt übereinstimmenden 10 Spalten erzeugen
plt.figure(figsize=(10, 6))

# WICHTIG: Wir übergeben das angepasste 10-Spalten-DataFrame X_test_summary
shap.summary_plot(shap_values_collapsed, X_test_summary, show=False)

plt.title(f"Globale Feature-Wichtigkeit ({best_model_name})", fontsize=14, pad=20)
plt.tight_layout()
plt.show()


In [ ]:

# --- LOKALE ANALYSE (Wasserfalldiagramm für die erste Testinstanz) ---
print(f"\n=== Wasserfalldiagramm für die erste Test-Instanz ===")
plt.figure(figsize=(10, 6))
# shap_values[0] nimmt die allererste Zeile aus deinem Testset
# max_display=10 begrenzt die Anzeige auf die 10 wichtigsten Einflussfaktoren dieser Instanz
shap.plots.waterfall(shap_values_collapsed[0], max_display=10, show=False)
plt.title(f"Wasserfalldiagramm für eine einzelne Vorhersage", fontsize=14, pad=20)
plt.tight_layout()
plt.show()


In [ ]:
#X['country'].unique()

## Vorhersagen und Speicherung

In [ ]:
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import Pipeline

'''
# Die beste Konfiguration definieren
best_model = Pipeline([
    ('poly', PolynomialFeatures(degree=2, interaction_only=True)),
    ('lr', LinearRegression())
])
'''


best_model = Pipeline([
    #("Random Forest (Deep & Random)", RandomForestRegressor(n_estimators=200, random_state=42, max_depth=15, min_samples_leaf=2, max_features='sqrt'))
    ('hgb', HistGradientBoostingRegressor(
        max_iter=200,          # Entspricht n_estimators bei XGBoost (Anzahl Bäume) 150
        max_depth=10,           # Maximale Tiefe der Bäume   5
        learning_rate=0.15,    # Lernrate / Schrittweite (Overfitting-Bremse)   0.05
        random_state=42
    ))
])


'''
best_model = Pipeline([
    ("Random Forest", RandomForestRegressor(n_estimators=200, random_state=42, max_depth=15, min_samples_leaf=2, max_features='sqrt'))
])
'''

# Die finale Pipeline inklusive Vorverarbeitung erstellen
final_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor), # Ihr bestehender ColumnTransformer
    ('regressor', best_model)       # Das polynomiale Modell
])

# Ein letztes Mal mit den Trainingsdaten füttern
final_pipeline.fit(X_train, y_train)



y_train_pred = final_pipeline.predict(X_train)
y_test_pred = final_pipeline.predict(X_test)

r2_train = r2_score(y_train, y_train_pred)
r2_test = r2_score(y_test, y_test_pred)

rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))

mae_train = mean_absolute_error(y_train, y_train_pred)
mae_test = mean_absolute_error(y_test, y_test_pred)

In [ ]:
# Beispiel: Ein neuer Datensatz mit zwei Personen/Messungen
neue_daten = pd.DataFrame({
    'outdoor_air_temperature': [10.7, 24.40],   # die ersten beiden Punkte aus X_test
    'air_temperature': [19.2, 26.7],
    'relative_humidity': [54.8, 50.3],
    'season': ['winter', 'summer'],     # Kategorische Spalte 1 (wird automatisch encodiert) - falsche Eingabe führt zum Ignorieren des Wertes!!!
    'country': ['uk', 'pakistan'],      # Kategorische Spalte 2
    'building_type': ['classroom', 'office'], # Kategorische Spalte 3
    'air_speed': [0.09, 0.1444],          # Die numerische Spalte (wird automatisch skaliert)
    'metabolic_rate': [1.2, 1.00683],
    'climate_zone': ['Temperate','Temperate'],
    'cooling_type': ['naturally ventilated', 'naturally ventilated']
    # Falls Sie noch weitere Spalten in feature_names hatten, müssen diese hier ebenfalls rein!
})

# Vorhersage berechnen
vorhersagen = final_pipeline.predict(neue_daten)

# Ergebnisse übersichtlich anzeigen
for i, vorhersage in enumerate(vorhersagen):
    print(f"Person {i+1}: Vorhergesagter Kleidungs-Isolationswert = {vorhersage:.2f} Clo")

#103425, 21857

Die Ergebnisse sollten: 0.55 und 0.77 sein!!! Berechnet 0.49 und 0.57 mit pol. Regression und 0.47 und 0.64 mit Random Forest.

Ziel: 0,75 und 0,98, statt 0,66 und 0,67

In [ ]:
import joblib

metrics = {
    "y_train_pred" : y_train_pred,
    "y_test_pred" : y_test_pred,
    "r2_train" : r2_train,
    "r2_test" : r2_test,
    "rmse_train" : rmse_train,
    "rmse_test" : rmse_test,
    "mae_train" : mae_train,
    "mae_test" : mae_test
}

regression = {
    "model" : pipeline_to_explain,  # hier steckt alles drinnen, inkl. dem Preprocessing
    "metrics" : metrics,
    "explainer": explainer,
    "shap_values_raw": shap_values_raw,
    "shap_values": shap_values_collapsed,
    "X_test_summary": X_test_summary
}

# Speicherpfad und Dateiname definieren
modell_dateiname = 'finales_regressions_modell_HistGradientBoosting_sk1_9_0.joblib'
#modell_dateiname = 'finales_regressions_modell_RandomForest.joblib'

# Die komplette Pipeline (inklusive Vorverarbeitung und Polynomen) speichern
#joblib.dump(regression, modell_dateiname)

print(f"✅ Das Modell wurde erfolgreich unter '{modell_dateiname}' gespeichert!")


In [ ]:
# Laden eines gespeicherten Modells
'''
import pandas as pd
import joblib

# 1. Das gespeicherte Modell aus der Datei laden
geladenes_modell = joblib.load('finales_regressions_modell.joblib')

# 2. Neue Daten bereitstellen (müssen als DataFrame mit den gleichen Spaltennamen vorliegen)
neue_daten = pd.DataFrame({
    'outdoor_air_temperature': [20.1],
    'air_temperature': [18.1],
    'relative_humidity': [60],
    'season': ['winter'],     # Kategorische Spalte 1 (wird automatisch encodiert)
    'country': ['Germany'],      # Kategorische Spalte 2
    'building_type': ['office'], # Kategorische Spalte 3
    'air_speed': [0.15]          # Die numerische Spalte (wird automatisch skaliert)
})

# 3. Direkt die Vorhersage berechnen
vorhersage = geladenes_modell.predict(neue_daten)
print(f"Vorhergesagter Wert: {vorhersage[0]:.2f} Clo")
'''

### Anomalie Regression über Vorhersage

In [ ]:
#df_vergleich_reg = pd.DataFrame(y_test.round(decimals=2).copy()).rename(columns={y_test.name: 'Ziel'})

df_vergleich_reg = X_test.copy()

df_vergleich_reg['Ziel'] = pd.DataFrame(y_test.round(decimals=2).copy())

df_vergleich_reg['berechnet'] = final_pipeline.predict(X_test).round(decimals=2)
df_vergleich_reg['delta'] = (df_vergleich_reg['Ziel'] - df_vergleich_reg['berechnet']).round(decimals=2)

print(f"Maximale Abweichung: {df_vergleich_reg['delta'].max()}")
print(f"Minimale Abweichung: {df_vergleich_reg['delta'].min()}")

display(df_vergleich_reg)

In [ ]:
#deltas = np.abs(y_test - y_pred)
deltas = np.abs(df_vergleich_reg['delta'])

# 2. Definiere die Grenze bei den 5% extremsten Abweichungen (95. Perzentil)
anomalie_grenze_95 = np.percentile(deltas, 95)
# Definiere die Grenze für "starke Anomalien" bei den obersten 1%
anomalie_grenze_99 = np.percentile(deltas, 99)

print(f"Ab einem Delta von {anomalie_grenze_95:.2f} clo liegt ein Datensatz in den Top 5% der Fehler.")
print(f"Ab einem Delta von {anomalie_grenze_99:.2f} clo liegt ein Datensatz in den Top 1% (Starke Anomalie).")

In [ ]:
df_regression_anomalie = df_vergleich_reg[np.abs(df_vergleich_reg['delta']) >= anomalie_grenze_99]
df_regression_anomalie['delta_abs'] = np.abs(df_regression_anomalie['delta'])

df_regression_anomalie['country'] = df['country']

display(df_regression_anomalie)

df_regression_anomalie.drop(columns=['delta']).to_csv("df_regression_anomalie.csv", index=False)

# Zusatzinformationen hinzufügen und als df abspeichern für Präsentation

In [ ]:
print(y.describe())

y.hist(bins=40)

In [ ]:
db_grosse_abw = df_vergleich_reg[df_vergleich_reg['delta']>1]
db_grosse_abw   #   8 Datensätze mit vermeindlich großer Anomalie!!!

Sehr interessant. 8 Werte mit sehr großen Abweichungen in den clo Werten. KONTEXTBEZOGENE ANOMALIEERKENNUNG


SHAP oder Featureimportance?

Vorhersagen ändern sich sicherlich stark, wenn man die Region einschränkt! In Indien wird bei gleicher Temperatur sicherlich andere Kleidung getragen als in Deutschland.

In [ ]:
sns.boxenplot(data=y)
plt.show()
sns.boxplot(data=y)
plt.show()

In [ ]:
'''
# 1. Namen der Features nach der ersten Vorverarbeitung (Preprocessor) holen
preprocessed_features = final_pipeline.named_steps['preprocessor'].get_feature_names_out()

# 2. Tiefer in den Regressor greifen, um die PolynomialFeatures und die Regression zu isolieren
inner_pipeline = final_pipeline.named_steps['regressor']
poly_step = inner_pipeline.named_steps['poly']
lr_step = inner_pipeline.named_steps['lr']

# 3. Die finalen polynomialen/Interaktions-Feature-Namen generieren
final_feature_names = poly_step.get_feature_names_out(input_features=preprocessed_features)

# 4. Koeffizienten auslesen
coefficients = lr_step.coef_
intercept = lr_step.intercept_

# 5. Alles übersichtlich in einer pandas Series zusammenführen
coef_overview = pd.Series(coefficients, index=final_feature_names)

# Ausgabe anzeigen
print(f"Achsenabschnitt (Intercept): {intercept}\n")
print("Koeffizienten der Features (sortiert nach Stärke des Einflusses):")
ausgabe =  coef_overview.reindex(coef_overview.abs().sort_values(ascending=False).index)
print(ausgabe)
'''

# Target Building Type

In [ ]:
df_ml_bt = df[['climate', 'air_temperature', 'relative_humidity', 'air_speed', 'clothing_ensemble_insulation', 'building_type']]
#df_ml_bt = df[['air_temperature', 'relative_humidity', 'air_speed', 'clothing_ensemble_insulation', 'building_type']]   # Problem mit Climate

In [ ]:
df_ml_bt_clean = df_ml_bt.dropna()

In [ ]:
df_ml_bt_clean.info()

In [ ]:
#print(df_ml_bt_clean['climate'].value_counts())

In [ ]:
print(f"Prozentuale Werteverteilung: \n {df_ml_bt_clean['building_type'].value_counts(normalize=True) * 100}")
print(f"Absolute Werteverteilung: \n {df_ml_bt_clean['building_type'].value_counts()}")

sns.histplot(data=df_ml_bt_clean['building_type'])

In [ ]:
# label encoding der Spalte buidling_type

from sklearn.preprocessing import LabelEncoder  # nur für targets

le = LabelEncoder()
df_ml_bt_clean['building_type_enc'] = le.fit_transform(df_ml_bt_clean['building_type'])


#df_test = le.inverse_transform(df_ml_bt_clean['building_type_enc'])


In [ ]:
'''
# label encoding für climate
# nicht zwingend notwendig für Random Forest und sorgt dafür, dass sehr viele Features entstehen!!!
# das führt zu Problemen bei der Feature_importance und vor allem bei SHAP!!!

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder

ct = ColumnTransformer(
    transformers=[
        ('nominal_features', OneHotEncoder(sparse_output=False, handle_unknown='ignore'), ['climate'])
    ],
    remainder='passthrough' # WICHTIG: Behält Rest unverändert bei!
)

df_ml_bt_clean_processed = ct.fit_transform(df_ml_bt_clean)

neue_spalten = list(ct.named_transformers_['nominal_features'].get_feature_names_out(['climate']))
restliche_spalten = [col for col in df_ml_bt_clean.columns if col != 'climate']

df_ml_bt_ready = pd.DataFrame(df_ml_bt_clean_processed, columns= neue_spalten +  restliche_spalten)

# Es entstehen 29 Spalten durch die unterschiedlichen Climate-Werte, das führt zu starken Problemen bei SHAP und macht die Analyse quasi unmöglich
'''


In [ ]:
liste = df_ml_bt_clean['climate'].value_counts().to_list
liste

In [ ]:
#'''
# OrdinalEncoder als Alternative für den One-Hot-Encoder

from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder

klima_reihenfolge = [
    'continental subarctic', 'warm-summerÂ humid continental', 
    'monsoon-influenced hot-summer humid continental', 'cold semi-arid', 
    'semi arid midlatitude', 'semi arid high altitude', 'subtropical highland', 
    'monsoon-influenced temperate oceanic', 'temperate oceanic', 'west coast marine', 
    'temperature marine', 'temperate', 'humid midlatitude', 'warm-summer mediterranean', 
    'mediterranean', 'hot-summer mediterranean', 'monsoon-influenced humid subtropical', 
    'humid subtropical', 'hot semi-arid', 'hot arid', 'desert (hot arid)', 'hot desert', 
    'tropical wet savanna', 'tropical savanna', 'tropical monsoon', 'tropical', 
    'tropical rainforest', 'wet equatorial'
]

ct = ColumnTransformer(
    transformers=[
        ('ordinal_features', OrdinalEncoder(categories=[klima_reihenfolge], handle_unknown='use_encoded_value', unknown_value=-1), ['climate'])
    ],
    remainder='passthrough'
)

# Transformiert die Daten (Achtung: Spaltenreihenfolge ändert sich!)
df_ml_bt_clean_processed = ct.fit_transform(df_ml_bt_clean)

# Die Spaltenreihenfolge nach dem ColumnTransformer:
# Zuerst die transformierte 'climate'-Spalte, danach der Rest
restliche_spalten = [col for col in df_ml_bt_clean.columns if col != 'climate']
neue_spalten = ['climate'] + restliche_spalten

df_ml_bt_ready = pd.DataFrame(df_ml_bt_clean_processed, columns= neue_spalten)
#'''
#df_ml_bt_ready = df_ml_bt_clean

In [ ]:
sns.pairplot(data=df_ml_bt_ready, hue='building_type')

In [ ]:
# feature engineering

#from sklearn.preprocessing import OrdinalEncoder
#from sklearn.preprocessing import TargetEncoder

#mapping = {'hot-summer mediterranean': 1, 'humid subtropical': 2, 'hot semi-arid': 3, 'tropical wet savanna': 4, 'mediterranean': 5, 'temperate oceanic': 6, 'tropical savanna': 7, 'warm-summer humid continental': 8, 'monsoon-influenced hot-summer humid continental': 9, 'tropical monsoon': 10, 'desert (hot arid)': 11, 'hot desert': 12, 'tropical': 13, 'monsoon-influenced humid subtropical': 14, 'subtropical highland': 15, 'warm-summer mediterranean': 16, 'west coast marine': 17, 'hot arid': 18, 'semi arid high altitude': 19, 'semi arid midlatitude': 20, 'temperature marine': 21, 'continental subarctic': 22, 'tropical rainforest': 23, 'wet equatorial':24, 'temperate': 25, 'monsoon-influenced temperate oceanic': 26, 'cold semi-arid':27, 'humid midlatitude': 28, 'subtropical hot and dry': 29}
#df_ml_bt_clean['climate'] = df_ml_bt_clean['climate'].replace(mapping)    # Encoding so für alle Algorithmen gut?


#mapping = {'office': 1, 'multifamily housing': 2, 'classroom': 3, 'senior center': 4, 'others': 5}
#df_ml_bt_clean['building_type'] = df_ml_bt_clean['building_type'].replace(mapping)    # Encoding so für alle Algorithmen gut?

In [ ]:
#df_ml_bt_clean_balanciert = df_ml_bt_clean.groupby('building_type').sample(n=500, random_state=42) # 6*400 Datensätze

In [ ]:
#print(df_ml_bt_clean['climate'].value_counts())

In [ ]:
# Korrelationsmatrix

corr_matrix = df_ml_bt_ready.drop(columns=['building_type']).corr()

# 2. Grafik (Heatmap) erstellen
plt.figure(figsize=(8, 6))

# sns.heatmap zeichnet das Diagramm
# annot=True schreibt die exakten Zahlenwerte in die Kästchen
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm',  # Blau = negative Korrelation, Rot = positive Korrelation
    fmt=".2f",        # Auf 2 Nachkommastellen runden
    linewidths=0.5,   # Dünne Trennlinien zwischen den Kästchen
    vmin=-1, vmax=1   # Die Farbskala exakt von -1 bis +1 begrenzen
)

plt.title('Korrelationsmatrix der Features', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

## Schneller Random Forest

In [ ]:
y = df_ml_bt_ready['building_type_enc'].astype(int)
X = df_ml_bt_ready.drop(columns=['building_type_enc', 'building_type'])

In [ ]:
'''
from imblearn.over_sampling import SMOTE

smote = SMOTE(random_state=42)
X_augmented, y_augmented = smote.fit_resample(X, y)
'''

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import train_test_split

# kein Scaler notwendig!

# Split bleibt identisch
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Random Forest Classifier (class_weight hilft bei unbalancierten Daten)
#model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight="balanced")
#model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight="balanced") # class_weight="balacened" für ungleiche Verteilung

# mit climate
model = RandomForestClassifier(n_estimators=100, max_depth=20, random_state=42, class_weight="balanced", n_jobs=-1, min_samples_leaf=5) # class_weight="balacened" für ungleiche Verteilung

# ohne climate
#model = RandomForestClassifier(n_estimators=100, max_depth=18, random_state=42, class_weight="balanced", n_jobs=-1, min_samples_leaf=6) # class_weight="balacened" für ungleiche Verteilung
model.fit(X_train, y_train)

#############

# Vorhersagen treffen
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

f1_train = f1_score(y_train, y_train_pred, average='macro')
f1_test = f1_score(y_test, y_test_pred, average='macro')

print("Random Forest")

# Automatische Interpretation der Lücke (Gap)
gap = f1_train - f1_test
print(f"f1_train: {f1_train}")
print(f"f1_test: {f1_test}")
print(f"Lücke (Gap) beim F1-Score: {gap:.2f}")

if gap > 0.15:
    print("⚠️ Warnung: Starkes Overfitting! Das Modell schneidet im Training deutlich besser ab als auf den Testdaten.")
elif gap > 0.05:
    print("💡 Hinweis: Leichtes Overfitting. Das ist bei Random Forests oft normal, behalten Sie es aber im Auge.")
else:
    print("✅ Optimal: Kein Overfitting. Das Modell generalisiert hervorragend auf neuen Daten.")
print("=========================\n")

#############

# Modell bewerten
accuracy = accuracy_score(y_test, y_test_pred)
print(f"Genauigkeit (Accuracy) - Random Forest: {accuracy:.2f}")
print("\nDetaillierter Klassifikationsbericht:")
print(classification_report(y_test, y_test_pred, zero_division=0))


# 0 = classroom
# 1 = multifamily housing
# 2 = office
# 3 = other
# 4 = senior center

In [ ]:
# Visualisierung (Feature Importance statt einzelnem Baum, da es viele Bäume gibt)
import pandas as pd
import matplotlib.pyplot as plt

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)
importances.plot(kind='bar', figsize=(10, 6))
plt.title("Feature Importances (Wichtigkeit der Merkmale)")
plt.show()

In [ ]:
'''
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay


###########################################################
# ggf. die Spalten durch Featurename ersetzen
#df_test = le.inverse_transform(df_ml_bt_clean['building_type_enc'])
###########################################################

# confusion matrix
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues")
plt.title("Random Forest - Anzahl")
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", normalize='true') # Zeilenweise (per true class)
plt.title("Random Forest - Zeilenweise (recall)")
plt.show()
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", normalize='pred') # Spaltenweise (per predicted class)
#ConfusionMatrixDisplay.from_predictions(y_test, y_pred, cmap="Blues", normalize='all') # Global (über den gesamten Datensatz)
plt.title("Random Forest - Spaltenweise (predict)")
plt.show()
'''

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

class_names = le.classes_

# confusion matrix - Anzahl
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, display_labels=class_names, cmap="Blues")
plt.title("Random Forest - Anzahl")
plt.xticks(rotation=90, ha='right') # Rotierte Labels auf der X-Achse
plt.tight_layout()                  # Verhindert, dass abgeschnittener Text entsteht
plt.show()

# confusion matrix - Zeilenweise (recall)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, display_labels=class_names, cmap="Blues", normalize='true')
plt.title("Random Forest - Zeilenweise (recall)")
plt.xticks(rotation=90, ha='right') # Rotierte Labels auf der X-Achse
plt.tight_layout()                  # Verhindert, dass abgeschnittener Text entsteht
plt.show()

# confusion matrix - Spaltenweise (predict)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, display_labels=class_names, cmap="Blues", normalize='pred')
plt.title("Random Forest - Spaltenweise (predict)")
plt.xticks(rotation=90, ha='right') # Rotierte Labels auf der X-Achse
plt.tight_layout()                  # Verhindert, dass abgeschnittener Text entsteht
plt.show()


In [ ]:
#import pickle
#
#with open('shap_values_object.pkl', 'rb') as f:
#    loaded_shap_object = pickle.load(f)

In [ ]:
# Funktioniert nicht bei one-hot-encoding, da dadurch zu viele Features entstehen!!!
import shap
import numpy as np

#X_test_sample = X_test.sample(n=1000, random_state=42)  # sonst sehr sehr lange Rechenzeit, da Baum sehr tief ist!!!

# 1. Explainer wie gewohnt erstellen
explainer = shap.TreeExplainer(model)


# 2. NEU: Direktes Aufrufen erzeugt das korrekte Explanation-Objekt
# (Berechnet die SHAP-Werte und behält Feature-Namen & Basiswerte)
#shap_values_object = explainer(X_test)  # sehr rechenintensiver Bereich - exponentiell zur Featureanzahl - auch die Tiefe des Baumes ist sehr entscheidend!!!

# alternativ
#shap_values_object = explainer(X_test.values, approximate=True, check_additivity=False)
shap_values_object = explainer(X_test, approximate=True, check_additivity=False)    # gröberer Wert, dafür extrem schnell

In [ ]:
import pickle

with open('shap_values_object.pkl', 'wb') as f:
    pickle.dump(shap_values_object, f)

In [ ]:
# 2. Umwandlung in das von SHAP erwartete Listen-Format für alle Klassen
# Extrahiert die Daten für jede der 6 Klassen separat in eine Liste
shap_all_classes = [shap_values_object.values[:, :, i] for i in range(len(model.classes_))]

# 3. Der komplette Gesamt-Plot
shap.summary_plot(
    shap_all_classes, 
    X_test, 
    plot_type="bar", 
    class_names=model.classes_  # Benennt die Legende nach Ihren echten Klassen
)

In [ ]:
# 3. Globale Übersicht (funktioniert weiterhin mit dem Objekt)
# shap.summary_plot(shap_values_object, X_test)
#shap.summary_plot(shap_values_object[:, :, 0], X_test)  # globale Betrachtung

# Violinenplots für unterschiedliche Klassen
#print("Klasse 5, Index 4")
#shap.summary_plot(shap_values_object[:, :, 4], X_test)
#print("Klasse 6, Index 5")
#shap.summary_plot(shap_values_object[:, :, 5], X_test)  # globale Betrachtung

for i in range(len(model.classes_)):
    print(f"Klasse mit Index: {i}")
    shap.summary_plot(shap_values_object[:, :, i], X_test.astype(float))  # globale Betrachtung

# 4. Erklärung der ersten Beobachtung (jetzt fehlerfrei)
# Bei Binär-Klassifikation / Regression:
# shap.plots.waterfall(shap_values_object[0])

'''
beobachtung = 0  # Die erste Zeile aus X_test
#ziel_klasse = 2  # Erklärung für die DRITTE Klasse Ihres Targets
ziel_klasse = 5  # Erklärung für die DRITTE Klasse Ihres Targets

shap.plots.waterfall(shap_values_object[beobachtung, :, ziel_klasse])
'''

In [ ]:
## 1. Explainer und SHAP-Werte erstellen
#explainer = shap.TreeExplainer(model)
#shap_values_object = explainer(X_test)

# 2. Parameter für die Analyse festlegen
#beobachtung_idx = 0  # Index der Zeile, die Sie erklären möchten
beobachtung_idx = 1  # Index der Zeile, die Sie erklären möchten

# 3. Die vorhergesagte Klasse für diese spezifische Zeile ermitteln
# model.predict() liefert das Label (z. B. "Klasse 3" oder Text)
vorhersage = model.predict(X_test.iloc[[beobachtung_idx]])[0]

# Falls Ihr Modell Klassenlabels (z.B. Strings) statt numerischer Indizes (0-5) nutzt,
# ermitteln wir hier den exakten numerischen Index der Klasse im Modell:
klasse_idx = np.where(model.classes_ == vorhersage)[0][0]

# 4. Textausgabe zur Kontrolle
print(f"Erklärung für Beobachtung {beobachtung_idx}")
print(f"Modell-Vorhersage: {vorhersage} (Klassen-Index im SHAP-Objekt: {klasse_idx})")

# 5. Waterfall-Plot exakt für die vorhergesagte Klasse aufrufen
shap.plots.waterfall(shap_values_object[beobachtung_idx, :, klasse_idx], show=False)    # Einzelvorhersage / Erklärung / Ein Datensatz wird erklärt
plt.title(f"Modell-Vorhersage: {vorhersage} (Klassen-Index im SHAP-Objekt: {klasse_idx}) für Datenpunkt {beobachtung_idx}")
plt.show()
# Unten E[f(X)] ist der Durchschnitt, der Erwartungswert für diese Klasseals Durchschnitt über den gesamten Trainingsdatensatz
# Nach oben hin Stück für Stück Berechnung des finalen Ergebnis

# z.B. für Zeile mit Index 0: 0.672 ist Wahrscheinlichkeit für den Klassenwert

## Alternative zu Random Forest!!!

In [ ]:
'''
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.preprocessing import OrdinalEncoder

# Strings in 0, 1, 2, 3, 4 umwandeln
oe = OrdinalEncoder()
X_encoded = oe.fit_transform(X[['mein_string_feature']])

# Modell aktivieren und kategoriales Feature explizit zuweisen
model = HistGradientBoostingClassifier(categorical_features=[0])
model.fit(X_encoded, y)
'''

# Cooling Type

In [ ]:
#df_cooling = df[['air_temperature', 'outdoor_air_temperature', 'relative_humidity', 'air_speed', 'radiant_temperature', 'globe_temperature', 'clothing_ensemble_insulation', 'cooling_type']]
#df_cooling = df[['air_temperature', 'outdoor_air_temperature', 'relative_humidity', 'air_speed', 'radiant_temperature', 'clothing_ensemble_insulation', 'cooling_type']]
#df_cooling = df[['air_temperature', 'outdoor_air_temperature', 'relative_humidity', 'air_speed', 'clothing_ensemble_insulation', 'cooling_type']]
df_cooling = df[['air_temperature', 'outdoor_air_temperature', 'relative_humidity', 'air_speed', 'clothing_ensemble_insulation', 'metabolic_rate', 'cooling_type']]

In [ ]:

df_cooling['cooling_type'] = df_cooling['cooling_type'].replace(['unknown', 'Unknown', 'UNKNOWN', 'nan', 'None'], np.nan)


df_cooling

In [ ]:
df_cooling_clean = df_cooling.dropna()

In [ ]:
df_cooling_clean.info()

In [ ]:
# cooling type -> label encoding

from sklearn.preprocessing import LabelEncoder  # nur für targets

le = LabelEncoder()
df_cooling_clean['cooling_type_enc'] = le.fit_transform(df_cooling_clean['cooling_type'])

# 0 - air conditioned
# 1 - mixed mode - könnte man auch erstmal entfernen
# 2 - naturally ventilated

In [ ]:
# Zeilen entfernen mit cooling Type unknown

#df_cooling_clean_filtered = df_cooling_clean[df_cooling_clean['cooling_type_enc'] != 0]
df_cooling_clean_filtered = df_cooling_clean.drop(columns=['cooling_type'])

In [ ]:
df_temp = df_cooling_clean.drop(columns=['cooling_type_enc'])
#display(df_temp) 
sns.pairplot(data=df_temp, hue='cooling_type', palette="Set2")
#sns.pairplot(data=df_cooling_clean_filtered, hue='cooling_type_enc', palette="Set2")
plt.show()
del df_temp

In [ ]:
# Korrelationsmatrix

corr_matrix = df_cooling_clean_filtered.corr()

# 2. Grafik (Heatmap) erstellen
plt.figure(figsize=(8, 6))

# sns.heatmap zeichnet das Diagramm
# annot=True schreibt die exakten Zahlenwerte in die Kästchen
sns.heatmap(
    corr_matrix, 
    annot=True, 
    cmap='coolwarm',  # Blau = negative Korrelation, Rot = positive Korrelation
    fmt=".2f",        # Auf 2 Nachkommastellen runden
    linewidths=0.5,   # Dünne Trennlinien zwischen den Kästchen
    vmin=-1, vmax=1   # Die Farbskala exakt von -1 bis +1 begrenzen
)

plt.title('Korrelationsmatrix der Features', fontsize=14, pad=15)
plt.tight_layout()
plt.show()

starke Korrelationen, alles über 0.8 hinterfragen, alles über 0.9 bereinigen => globe Temperature zunächst raus, ggf. auch radient temp

In [ ]:
sns.histplot(data=df_cooling_clean_filtered['cooling_type_enc'], discrete=True, shrink=0.8)
plt.gca().xaxis.set_major_locator(plt.MaxNLocator(integer=True))
plt.xlim([-0.5,2.5])
plt.show()

In [ ]:
y = df_cooling_clean_filtered['cooling_type_enc'].astype(int)
X = df_cooling_clean_filtered.drop(columns=['cooling_type_enc'])

nur kontinuierliche Features!!!

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import PowerTransformer, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.metrics import classification_report, accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline

# 1. Split (mit Stratify für unbalancierte Daten)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Spaltennamen definieren
#schiefe_features = ['air_speed', 'clothing_ensemble_insulation']
schiefe_features = ['air_speed', 'metabolic_rate', 'clothing_ensemble_insulation']
#normale_features = ['air_temperature', 'relative_humidity', 'outdoor_air_temperature', 'globe_temperature']
normale_features = ['air_temperature', 'relative_humidity', 'outdoor_air_temperature']

# 2. Preprocessor bauen
preprocessor = ColumnTransformer(
    transformers=[
        # PowerTransformer normalisiert UND skaliert die schiefen Spalten automatisch
        ('schief', PowerTransformer(method='yeo-johnson'), schiefe_features),
        # Normale Features brauchen nur den StandardScaler
        ('normal', StandardScaler(), normale_features)
    ]
)

# 3. Modelle in einer Schleife testen
modelle = {
    "Logistische Regression": LogisticRegression(max_iter=100, class_weight='balanced', penalty='l2', random_state=42),
    "Logistische Regression (Poly)": make_pipeline(StandardScaler(), PolynomialFeatures(degree=2, interaction_only=True, include_bias=False), LogisticRegression(max_iter=1000, class_weight='balanced', penalty='l2', random_state=42)),
    
    # Bäume stört der PowerTransformer nicht, sie funktionieren damit genauso gut
    #"Decision Tree": DecisionTreeClassifier(max_depth=6, random_state=42, class_weight='balanced'),
    "Decision Tree": DecisionTreeClassifier(max_depth=8, random_state=42, class_weight='balanced'),

    "kNN": KNeighborsClassifier(n_neighbors=60, weights="uniform"),
    #"Random Forest": RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42, class_weight="balanced", n_jobs=-1), # max_depth=5
    #"Random Forest": RandomForestClassifier(n_estimators=200, max_depth=10, min_samples_leaf=20, random_state=42, class_weight="balanced", n_jobs=-1), # max_depth=5
    #"Random Forest": RandomForestClassifier(n_estimators=1000, max_depth=20, random_state=42, class_weight="balanced", n_jobs=-1,min_samples_leaf=6),
    
    #"Random Forest": RandomForestClassifier(n_estimators=500, max_depth=20, random_state=42, class_weight="balanced", n_jobs=-1,min_samples_leaf=6), # sehr gute Ergebnisse, aber beim Export sehr große Datei
    #"Random Forest": RandomForestClassifier(n_estimators=200, max_depth=30, random_state=42, class_weight="balanced", n_jobs=-1,min_samples_leaf=6),

    # letztes Modell 
    "Random Forest": RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, class_weight="balanced", n_jobs=-1,min_samples_leaf=6),
    
    # oben gleich letztes Modell
    "HistGradientBoosting": HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05, max_depth=20, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=30.0, early_stopping=True, n_iter_no_change=15, validation_fraction=0.1, class_weight='balanced', random_state=42) #
    
    #"HistGradientBoosting": HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05, max_leaf_nodes=45, min_samples_leaf=15, early_stopping=True, n_iter_no_change=15, class_weight='balanced', random_state=42) #
    #"HistGradientBoosting": HistGradientBoostingClassifier(max_iter=500, learning_rate=0.05, max_depth=25, max_leaf_nodes=15, min_samples_leaf=100, l2_regularization=30.0, early_stopping=True, n_iter_no_change=15, validation_fraction=0.1, class_weight='balanced', random_state=42) #
}

trainierte_pipelines = {}

for name, model in modelle.items():
    # Pipeline für das aktuelle Modell erstellen
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Trainieren (Transformer lernt NUR von X_train)
    pipeline.fit(X_train, y_train)
    
    # Pipeline in unserem Dictionary für spätere Auswertungen sichern
    trainierte_pipelines[name] = pipeline

    # Vorhersagen (Transformer wendet gelerntes Wissen auf X_test an)
    y_pred = pipeline.predict(X_test)
    
    print(f"\n===== {name} =====")
    print(f"Accuracy: {accuracy_score(y_test, y_pred):.2f}")
    print(classification_report(y_test, y_pred))

    #############################################
    # Automatische Interpretation der Lücke (Gap)

    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    f1_train = f1_score(y_train, y_train_pred, average='macro')
    f1_test = f1_score(y_test, y_test_pred, average='macro')

    gap = f1_train - f1_test
    print(f"f1_train: {f1_train}")
    print(f"f1_test: {f1_test}")
    print(f"Lücke (Gap) beim F1-Score: {gap:.2f}")

    if gap > 0.15:
        print("⚠️ Warnung: Starkes Overfitting! Das Modell schneidet im Training deutlich besser ab als auf den Testdaten.")
    elif gap > 0.05:
        print("💡 Hinweis: Leichtes Overfitting. Das ist bei Random Forests oft normal, behalten Sie es aber im Auge.")
    else:
        print("✅ Optimal: Kein Overfitting. Das Modell generalisiert hervorragend auf neuen Daten.")
    print("=========================\n")

    #############################################

In [ ]:
# Dictionary, um die Ergebnisse für die Grafik zu speichern
macro_f1_results = {}

# Die Schleife trainiert die Modelle und speichert den Macro F1-Score
for name, model in modelle.items():
    pipeline = Pipeline(steps=[
        ('preprocessor', preprocessor),
        ('classifier', model)
    ])
    
    # Trainieren & Vorhersagen
    pipeline.fit(X_train, y_train)
    y_pred = pipeline.predict(X_test)
    
    # Einheitlichen Macro F1-Score berechnen und speichern
    score = f1_score(y_test, y_pred, average='macro')
    macro_f1_results[name] = score

# --- Grafik erstellen ---
plt.figure(figsize=(10, 6))

# Farben für die einzelnen Balken definieren
farben = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']

# Balkendiagramm zeichnen
bars = plt.bar(macro_f1_results.keys(), macro_f1_results.values(), color=farben, edgecolor='black', width=0.6)

# Werte oben auf die Balken schreiben
for bar in bars:
    height = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2.0, height + 0.01, f'{height:.2f}', ha='center', va='bottom', fontsize=11, fontweight='bold')

# Diagramm-Details anpassen
plt.title('Modellvergleich: Macro F1-Score (Thermal Sensation 3 Werte)', fontsize=14, pad=15)
plt.ylabel('Macro F1-Score', fontsize=12)
plt.ylim(0, 1.1)  # Platz für die Textlabels oben lassen
plt.grid(axis='y', linestyle='--', alpha=0.7)

# Grafik anzeigen
plt.tight_layout()
plt.xticks(rotation=90)
plt.show()

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split

# kein Scaler notwendig!

# Split bleibt identisch
#X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# Random Forest Classifier (class_weight hilft bei unbalancierten Daten)
#model = RandomForestClassifier(n_estimators=100, max_depth=6, random_state=42, class_weight="balanced")
#model = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=42, class_weight="balanced") # class_weight="balacened" für ungleiche Verteilung
#model = RandomForestClassifier(n_estimators=1000, max_depth=4, random_state=42, class_weight="balanced", n_jobs=-1) # kein overfitting

# manuell bestes Modell
model = RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, class_weight="balanced", n_jobs=-1,min_samples_leaf=6) # class_weight="balacened" für ungleiche Verteilung

# aus GridSearchCV, aber mit 0,9927 sehr suspekt, es lernt die Trainingsdaten zu sehr auswendig!
#model = RandomForestClassifier(n_estimators=200, max_depth=None, random_state=42, class_weight="balanced", n_jobs=-1,min_samples_leaf=2) # class_weight="balacened" für ungleiche Verteilung

model.fit(X_train, y_train)

#############

# Vorhersagen treffen
y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

f1_train = f1_score(y_train, y_train_pred, average='macro')
f1_test = f1_score(y_test, y_test_pred, average='macro')

# Automatische Interpretation der Lücke (Gap)
gap = f1_train - f1_test
print(f"f1_train: {f1_train}")
print(f"f1_test: {f1_test}")
print(f"Lücke (Gap) beim F1-Score: {gap:.2f}")

if gap > 0.15:
    print("⚠️ Warnung: Starkes Overfitting! Das Modell schneidet im Training deutlich besser ab als auf den Testdaten.")
elif gap > 0.05:
    print("💡 Hinweis: Leichtes Overfitting. Das ist bei Random Forests oft normal, behalten Sie es aber im Auge.")
else:
    print("✅ Optimal: Kein Overfitting. Das Modell generalisiert hervorragend auf neuen Daten.")
print("=========================\n")

#############

# Modell bewerten
print("Random Forest")
accuracy = accuracy_score(y_test, y_test_pred)
print(f"Genauigkeit (Accuracy) - Random Forest: {accuracy:.2f}")
print("\nDetaillierter Klassifikationsbericht:")
print(classification_report(y_test, y_test_pred, zero_division=0))

In [ ]:
db_vergleich = X_test.copy()
db_vergleich['cooling_type_real'] = y_test
db_vergleich['cooling_type_pred'] = y_test_pred


In [ ]:
neue_daten = pd.DataFrame({
    'air_temperature': [22.0],   # die ersten beiden Punkte aus X_test
    'outdoor_air_temperature': [24.5],
    'relative_humidity': [62],
    'air_speed': [0.00],
    'clothing_ensemble_insulation': [0.65],
    'metabolic_rate': [1.4]
})
ergebnis = model.predict(neue_daten)
ergebnis

In [ ]:
# Visualisierung (Feature Importance statt einzelnem Baum, da es viele Bäume gibt)
import pandas as pd
import matplotlib.pyplot as plt

importances = pd.Series(model.feature_importances_, index=X.columns).sort_values(ascending=False)
print(importances)
importances.plot(kind='bar', figsize=(10, 6))
plt.title("Feature Importances (Wichtigkeit der Merkmale)")
plt.show()

In [ ]:
from sklearn.metrics import confusion_matrix
from sklearn.metrics import ConfusionMatrixDisplay
import matplotlib.pyplot as plt

#class_names = le.classes_

# confusion matrix - Anzahl
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, cmap="Blues")
plt.title("Random Forest - Anzahl")
#plt.xticks(rotation=90, ha='right') # Rotierte Labels auf der X-Achse
plt.tight_layout()                  # Verhindert, dass abgeschnittener Text entsteht
plt.grid(False)
plt.show()

# confusion matrix - Zeilenweise (recall)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, cmap="Blues", normalize='true')
plt.title("Random Forest - Zeilenweise (recall)")
#plt.xticks(rotation=90, ha='right') # Rotierte Labels auf der X-Achse
plt.tight_layout()                  # Verhindert, dass abgeschnittener Text entsteht
plt.grid(False)
plt.show()

# confusion matrix - Spaltenweise (predict)
ConfusionMatrixDisplay.from_predictions(y_test, y_test_pred, cmap="Blues", normalize='pred')
plt.title("Random Forest - Spaltenweise (predict)")
#plt.xticks(rotation=90, ha='right') # Rotierte Labels auf der X-Achse
plt.tight_layout()                  # Verhindert, dass abgeschnittener Text entsteht
plt.grid(False)
plt.show()


In [ ]:
# Funktioniert nicht bei one-hot-encoding, da dadurch zu viele Features entstehen!!!
import shap
import numpy as np

#X_test_sample = X_test.sample(n=1000, random_state=42)  # sonst sehr sehr lange Rechenzeit, da Baum sehr tief ist!!!

# 1. Explainer wie gewohnt erstellen
explainer = shap.TreeExplainer(model)


# 2. NEU: Direktes Aufrufen erzeugt das korrekte Explanation-Objekt
# (Berechnet die SHAP-Werte und behält Feature-Namen & Basiswerte)
#shap_values_object = explainer(X_test)  # sehr rechenintensiver Bereich - exponentiell zur Featureanzahl - auch die Tiefe des Baumes ist sehr entscheidend!!!

# alternativ
shap_values_object = explainer(X_test.values, approximate=True, check_additivity=False)
#shap_values_object = explainer(X_test, approximate=True, check_additivity=False)    # gröberer Wert, dafür extrem schnell

In [ ]:
# 2. Umwandlung in das von SHAP erwartete Listen-Format für alle Klassen
# Extrahiert die Daten für jede der 6 Klassen separat in eine Liste
shap_all_classes = [shap_values_object.values[:, :, i] for i in range(len(model.classes_))]

# 3. Der komplette Gesamt-Plot
shap.summary_plot(
    shap_all_classes, 
    X_test, 
    plot_type="bar", 
    class_names=model.classes_  # Benennt die Legende nach Ihren echten Klassen
)

In [ ]:
# 3. Globale Übersicht (funktioniert weiterhin mit dem Objekt)
# shap.summary_plot(shap_values_object, X_test)
#shap.summary_plot(shap_values_object[:, :, 0], X_test)  # globale Betrachtung

# Violinenplots für unterschiedliche Klassen
#print("Klasse 5, Index 4")
#shap.summary_plot(shap_values_object[:, :, 4], X_test)
#print("Klasse 6, Index 5")
#shap.summary_plot(shap_values_object[:, :, 5], X_test)  # globale Betrachtung

for i in range(len(model.classes_)):
    print(f"Klasse mit Index: {i}")
    shap.summary_plot(shap_values_object[:, :, i], X_test)  # globale Betrachtung

# 4. Erklärung der ersten Beobachtung (jetzt fehlerfrei)
# Bei Binär-Klassifikation / Regression:
# shap.plots.waterfall(shap_values_object[0])

'''
beobachtung = 0  # Die erste Zeile aus X_test
#ziel_klasse = 2  # Erklärung für die DRITTE Klasse Ihres Targets
ziel_klasse = 5  # Erklärung für die DRITTE Klasse Ihres Targets

shap.plots.waterfall(shap_values_object[beobachtung, :, ziel_klasse])
'''

In [ ]:
## 1. Explainer und SHAP-Werte erstellen
#explainer = shap.TreeExplainer(model)
#shap_values_object = explainer(X_test)

# 2. Parameter für die Analyse festlegen
#beobachtung_idx = 0  # Index der Zeile, die Sie erklären möchten
beobachtung_idx = 1  # Index der Zeile, die Sie erklären möchten

# 3. Die vorhergesagte Klasse für diese spezifische Zeile ermitteln
# model.predict() liefert das Label (z. B. "Klasse 3" oder Text)
vorhersage = model.predict(X_test.iloc[[beobachtung_idx]])[0]

# Falls Ihr Modell Klassenlabels (z.B. Strings) statt numerischer Indizes (0-5) nutzt,
# ermitteln wir hier den exakten numerischen Index der Klasse im Modell:
klasse_idx = np.where(model.classes_ == vorhersage)[0][0]

shap_values_object.feature_names = list(X_test.columns)

# 4. Textausgabe zur Kontrolle
print(f"Erklärung für Beobachtung {beobachtung_idx}")
print(f"Modell-Vorhersage: {vorhersage} (Klassen-Index im SHAP-Objekt: {klasse_idx})")

# 5. Waterfall-Plot exakt für die vorhergesagte Klasse aufrufen
shap.plots.waterfall(shap_values_object[beobachtung_idx, :, klasse_idx], show=False)    # Einzelvorhersage / Erklärung / Ein Datensatz wird erklärt
plt.title(f"Modell-Vorhersage: {vorhersage} (Klassen-Index im SHAP-Objekt: {klasse_idx}) für Datenpunkt {beobachtung_idx}")
plt.show()
# Unten E[f(X)] ist der Durchschnitt, der Erwartungswert für diese Klasseals Durchschnitt über den gesamten Trainingsdatensatz
# Nach oben hin Stück für Stück Berechnung des finalen Ergebnis

# z.B. für Zeile mit Index 0: 0.672 ist Wahrscheinlichkeit für den Klassenwert

In [ ]:
# SHAP Dependence Plot

gewaehlte_klasse = 0

# Diagramm aufbauen
shap.plots.scatter(
    shap_values_object[:, "air_temperature", gewaehlte_klasse], 
    color=shap_values_object[:, :, gewaehlte_klasse], show=False
)
plt.title("SHAP Dependence plot - air_conditioned")
plt.show()

gewaehlte_klasse = 1

# Diagramm aufbauen
shap.plots.scatter(
    shap_values_object[:, "metabolic_rate", gewaehlte_klasse], 
    color=shap_values_object[:, :, gewaehlte_klasse], show=False
)
plt.title("SHAP Dependence plot - mixed mode")
plt.show()

# Diagramm aufbauen
shap.plots.scatter(
    shap_values_object[:, "outdoor_air_temperature", gewaehlte_klasse], 
    color=shap_values_object[:, :, gewaehlte_klasse], show=False
)
plt.title("SHAP Dependence plot - mixed mode")
plt.show()

gewaehlte_klasse = 2

# Diagramm aufbauen
shap.plots.scatter(
    shap_values_object[:, "outdoor_air_temperature", gewaehlte_klasse], 
    color=shap_values_object[:, :, gewaehlte_klasse], show=False    # zweites Feature wird über den Interaktionsindex ausgewählt
)
plt.title("SHAP Dependence plot - naturally ventilated")
plt.show()

## Test Pruning

In [ ]:
'''
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

# 1. Maximalen Baum wachsen lassen
clf = DecisionTreeClassifier(random_state=42)
clf.fit(X_train, y_train)

# 2. Die möglichen Alpha-Werte für diesen Baum ermitteln
path = clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas, impurities = path.ccp_alphas, path.impurities

clfs = []
for ccp_alpha in ccp_alphas:
    clf = DecisionTreeClassifier(random_state=42, ccp_alpha=ccp_alpha)
    clf.fit(X_train, y_train)
    clfs = [] # (In der Praxis fügt man sie hier einer Liste hinzu)

# Finaler, optimal gestutzter Baum
final_clf = DecisionTreeClassifier(random_state=42, ccp_alpha=0.015) # Beispielwert
final_clf.fit(X_train, y_train)

# Vorhersagen treffen
y_pred = final_clf.predict(X_test)

# Modell bewerten
print("Random Forest")
accuracy = accuracy_score(y_test, y_pred)
print(f"Genauigkeit (Accuracy) - Random Forest: {accuracy:.2f}")
print("\nDetaillierter Klassifikationsbericht:")
print(classification_report(y_test, y_pred, zero_division=0))
'''

### GridSearch

In [ ]:
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, f1_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline

# 1. Basis-Pipeline definieren
base_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),  # Nutzt deinen bestehenden preprocessor
        (
            "classifier",
            RandomForestClassifier(
                random_state=42, class_weight="balanced", n_jobs=-1
            ),
        ),
    ]
)

# 2. Hyperparameter-Gitter (Param_Grid) für den Random Forest definieren
param_grid = {
    # "classifier__n_estimators": [50, 100, 200],  # Anzahl der Bäume
    # "classifier__max_depth": [6, 12, 20, None,],  # Maximale Tiefe (None = unbegrenzt)
    # "classifier__min_samples_leaf": [2, 6, 12, ],  # Verhindert Overfitting an Kanten
    # "classifier__max_features": ["sqrt", "log2",],  # Anzahl der Features pro Split

    "classifier__n_estimators": [8, 12, 15],  # Anzahl der Bäume
    "classifier__max_depth": [10, 20, 30, None,],  # Maximale Tiefe (None = unbegrenzt)
    #"classifier__min_samples_leaf": [2, 6, 12, ],  # Verhindert Overfitting an Kanten
    "classifier__max_features": ["sqrt"],  # Anzahl der Features pro Split
}

# 3. StratifiedKFold für unbalancierte Klassifikationsdaten einrichten
# Sichert ab, dass in jedem der 5 Splits das Verhältnis der Kühlungstypen identisch bleibt
cv_strategy = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

# 4. GridSearchCV konfigurieren
grid_search = GridSearchCV(
    estimator=base_pipeline,
    param_grid=param_grid,
    cv=cv_strategy,
    scoring="f1_macro",  # <--- Optimiert auf deinen einheitlichen Macro F1-Score
    n_jobs=-1,  # Nutzt alle CPU-Kerne für parallele Suche
    verbose=1,
)

# Suche starten
grid_search.fit(X_train, y_train)

print("\n--- Grid Search abgeschlossen ---")
print(f"Beste Hyperparameter: {grid_search.best_params_}")
print(f"Bester CV-Score (Macro F1-Score): {grid_search.best_score_:.4f}\n")


# --- 5. EVALUATION DER BESTEN PIPELINE ---

# Wir packen das beste gefundene Modell in dein models-Dictionary
models = {
    "Random Forest (Optimized via GridSearch)": grid_search.best_estimator_
}

# Schleife über deine Modelle
for name, pipeline in models.items():

    # Vorhersagen für Training und Test generieren
    y_train_pred = pipeline.predict(X_train)
    y_test_pred = pipeline.predict(X_test)

    # F1-Scores berechnen
    f1_train = f1_score(y_train, y_train_pred, average="macro")
    f1_test = f1_score(y_test, y_test_pred, average="macro")

    # Automatische Interpretation der Lücke (Gap)
    gap = f1_train - f1_test

    print(f"=== {name} ===")
    print(f"  f1_train: {f1_train:.4f}")
    print(f"  f1_test:  {f1_test:.4f}")
    print(f"  Lücke (Gap) beim F1-Score: {gap:.2f}")

    # Deine exakte Overfitting-Logik ausgeben
    if gap > 0.15:
        print(
            "  ⚠️ Warnung: Starkes Overfitting! Das Modell schneidet im Training deutlich besser ab als auf den Testdaten."
        )
    elif gap > 0.05:
        print(
            "  💡 Hinweis: Leichtes Overfitting. Das ist bei Random Forests oft normal, behalten Sie es aber im Auge."
        )
    else:
        print(
            "  ✅ Optimal: Kein Overfitting. Das Modell generalisiert hervorragend auf neuen Daten."
        )
    print("-" * 40)

    # Modell final bewerten
    accuracy = accuracy_score(y_test, y_test_pred)
    print(f"Genauigkeit (Accuracy) - {name}: {accuracy:.2f}")
    print("\nDetaillierter Klassifikationsbericht:")
    print(classification_report(y_test, y_test_pred, zero_division=0))
    print("=========================\n")

# läuft 4:30 min

In [ ]:
neue_daten = pd.DataFrame({
    'air_temperature': [22.0],   # die ersten beiden Punkte aus X_test
    'outdoor_air_temperature': [24.5],
    'relative_humidity': [62],
    'air_speed': [0.00],
    'clothing_ensemble_insulation': [0.65],
    'metabolic_rate': [1.4]
})
ergebnis = model.predict(neue_daten)
ergebnis

## Abspeichern des Modells

In [ ]:

import joblib

metrics = {
    "y_test" : y_test,
    "y_train" : y_train,
    "y_train_pred" : y_train_pred,
    "y_test_pred" : y_test_pred,
    "f1_train" : f1_train,
    "f1_test" : f1_test,
}

regression = {
    "model" : model,  # hier steckt alles drinnen, inkl. dem Preprocessing
    "metrics" : metrics,
    "explainer": explainer,
    "shap_values": shap_values_object,
    "LabelEncoder": le
}

# Speicherpfad und Dateiname definieren
modell_dateiname = 'finales_klassifikations_modell_RandomForest.joblib'

# Die komplette Pipeline (inklusive Vorverarbeitung und Polynomen) speichern
#joblib.dump(regression, modell_dateiname) # 417mb groß wegen der 500 Bäume und der Tiefe 167mb , sowie der SHAP-Analyse 244mb
#joblib.dump(regression, modell_dateiname, compress=3) 

print(f"✅ Das Modell wurde erfolgreich unter '{modell_dateiname}' gespeichert!")


In [ ]:
f1_train

In [ ]:
model

## CalibratedClassiverCV

In [ ]:
'''# Man setzt das calibrated_model immer dann ein, wenn die Höhe der Wahrscheinlichkeit eine geschäftliche oder physikalische Bedeutung hat.
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import RandomForestClassifier

# 1. Basis-Modell definieren
rf = RandomForestClassifier(random_state=42)

# 2. Kalibrierung vorschalten (nutzt intern Cross-Validation)
calibrated_model = CalibratedClassifierCV(estimator=rf, method='sigmoid', cv=5)
calibrated_model.fit(X_train, y_train)

# Dieses 'calibrated_model' speicherst du dann wie gewohnt in deiner .joblib Datei ab!
'''

In [ ]:
'''
import numpy as np
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score

# 1. Berechne den Pruning-Pfad auf den Trainingsdaten
path = clf.cost_complexity_pruning_path(X_train, y_train)
ccp_alphas = path.ccp_alphas

# Wir lassen das allerletzte Alpha weg, weil es den Baum komplett löscht (nur 1 Wurzelknoten)
ccp_alphas = ccp_alphas[:-1]

best_alpha = 0
best_f1 = 0

# 2. Schleife über alle möglichen Alphas, um das beste zu finden
for alpha in ccp_alphas:
    # Temporären Baum mit aktuellem Alpha trainieren
    test_clf = DecisionTreeClassifier(random_state=42, ccp_alpha=alpha)
    test_clf.fit(X_train, y_train)
    
    # Vorhersage auf Testdaten
    y_pred = test_clf.predict(X_test)
    current_f1 = f1_score(y_test, y_pred, average='macro')
    
    # Wenn dieser F1-Score besser ist als der bisherige Bestwert, merken!
    if current_f1 > best_f1:
        best_f1 = current_f1
        best_alpha = alpha

print(f"Das mathematisch beste Alpha ist: {best_alpha}")
print(f"Der damit erreichte beste Macro F1-Score ist: {best_f1}")

# 3. Finalen Baum mit dem optimalen Alpha trainieren
final_clf = DecisionTreeClassifier(random_state=42, ccp_alpha=best_alpha)
final_clf.fit(X_train, y_train)

# Vorhersagen treffen
y_pred = final_clf.predict(X_test)

# Modell bewerten
print("Random Forest")
accuracy = accuracy_score(y_test, y_pred)
print(f"Genauigkeit (Accuracy) - Random Forest: {accuracy:.2f}")
print("\nDetaillierter Klassifikationsbericht:")
print(classification_report(y_test, y_pred, zero_division=0))
'''

In [ ]:
'''

import pandas as pd
import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_regression
from sklearn.linear_model import LinearRegression

# 1. Beispiel-Daten erstellen
data = pd.DataFrame({
    'alter': [25, 47, 31, 22, np.nan],
    'einkommen': [50000, 80000, 62000, 35000, 95000],
    'stadt': ['Berlin', 'München', 'Berlin', 'Köln', 'München']
})
X = data.drop(columns=['einkommen'])
y = data['einkommen']

# 2. Definition der automatischen Schritte für numerische Daten
numeric_features = ['alter']
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),  # Fehlende Werte ersetzen
    ('poly', PolynomialFeatures(degree=2, include_bias=False)), # Automatische Interaktionen (X², X*Y)
    ('scaler', StandardScaler())  # Standardisierung
])

# 3. Definition der automatischen Schritte für kategorische Daten
categorical_features = ['stadt']
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('onehot', OneHotEncoder(handle_unknown='ignore')) # Automatische Dummy-Variablen
])

# 4. Transformatoren zusammenführen
preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ])

# 5. Die finale, automatisierte Pipeline (Preprocessing + Feature Selection + Modell)
auto_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('feature_selection', SelectKBest(score_func=f_regression, k='all')), # Automatische Auswahl der besten Features
    ('model', LinearRegression())
])

# 6. Automatisches Training und Feature Engineering starten
auto_pipeline.fit(X, y)


'''